# 1. Import and Hardware Setup

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset

import matplotlib.pyplot as plt

!pip install tqdm ipywidgets -q
from tqdm.auto import tqdm

import math


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
DATA_PATH = './data'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


# 2. Hyperparameters

In [3]:
def _make_divisible(old_ch, divisor=8, min_ch=None):
    if min_ch is None:
        min_ch = divisor
    # Calculate new channels
    new_ch = max(min_ch, int(old_ch + divisor // 2) // divisor * divisor)

    # Check if the new channels drop too much
    if new_ch < old_ch * 0.9:
        new_ch += divisor
    return new_ch

def get_dropout_rate(phi: float, min_rate: float = 0.2, max_rate: float = 0.5, max_phi: float = 7.0) -> float:
    rate = min_rate + (phi / max_phi) * (max_rate - min_rate)
    return rate

In [ ]:
PHI_VALUES = {
            "EfficientNet-B0": 0,
            "EfficientNet-B1": 1,
            "EfficientNet-B2": 2,
            "EfficientNet-B3": 3,
            "EfficientNet-B4": 4,
            "EfficientNet-B5": 5,
            "EfficientNet-B6": 6,
            "EfficientNet-B7": 7,
        }

MODEL_NAME = "EfficientNet-B0"
PHI = PHI_VALUES[MODEL_NAME]
ALPHA = 1.2
BETA = 1.1
GAMMA = 1.15

IMG_SIZE = _make_divisible(int(224 * (GAMMA**PHI)))
DEPTH_FACTOR = ALPHA ** PHI
WIDTH_FACTOR = BETA ** PHI
DROPOUT = get_dropout_rate(PHI)

0.2


In [ ]:
IN_CHANNELS = 3
BATCH_SIZE = 128
NUM_CLASSES = 101

EPOCHS = 150


# 3. Data Preparation

# 4. Model Architecture

<div style="display:flex; gap:10px; align-items:flex-start">
  <img src="figures/EfficientNet-B0.png" alt="EN-B0" style="width:55%; height:auto; object-fit:contain;"/>
  <img src="figures/EfficientNet-steps.png" alt="EN-steps" style="width:45%; height:auto; object-fit:contain;"/>
</div>

In [ ]:
class DropPath(nn.Module):
    def __init__(self, survival_prob: float = 0.8):
        super().__init__()
        self.survival_prob = survival_prob
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.survival_prob == 1.0:
            return x
        
        batch_size = x.shape[0]
        noise = torch.empty(batch_size, 1, 1, 1, device=x.device)
        noise.bernoulli_(self.survival_prob)
        
        if self.survival_prob > 0:
            noise.div_(self.survival_prob)
        
        return x * noise

class ConvBNAct(nn.Sequential):
    def __init__(self, in_channels, out_channels, kernel_size=1, stride=1, groups=1, activation=nn.SiLU):
        padding = (kernel_size - 1) // 2
        activation = (
                nn.Identity()
                if activation == nn.Identity
                else nn.SiLU(inplace=True)
            )
        super().__init__(
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                groups=groups,
                bias=False),
            nn.BatchNorm2d(out_channels),
            activation,
        )

class SqueezeExcitation(nn.Module):
    def __init__(self, in_channels, squeeze_factor=4):
        super().__init__()
        squeezed_channels = _make_divisible(in_channels // squeeze_factor, divisor=8)
        self.block = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, squeezed_channels, kernel_size=1),
            nn.SiLU(inplace=True),
            nn.Conv2d(squeezed_channels, in_channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.block(x)


class MBConv(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        expand_ratio=1,
        kernel_size=1,
        stride=1,
        survival_prob=0.8,
    ):
        super().__init__()
        hidden_channels = in_channels * expand_ratio
        self.use_res_connect = stride == 1 and in_channels==out_channels
        self.drop_path = DropPath(survival_prob) if self.use_res_connect else nn.Identity()
        
        layers = []
        
        # Expansion layer
        if expand_ratio != 1:
            layers.append(
                ConvBNAct(
                    in_channels=in_channels,
                    out_channels=hidden_channels,
                    kernel_size=1,
                    stride=1,
                    groups=1,
                    activation=nn.SiLU,
                )
            )
        
        # Depthwise layer
        layers.append(
            ConvBNAct(
                in_channels=hidden_channels,
                out_channels=hidden_channels,
                kernel_size=kernel_size,
                stride=stride,
                groups=hidden_channels,
                activation=nn.SiLU,
            )
        )
        
        # Squeeze and Excite
        layers.append(SqueezeExcitation(hidden_channels))
        
        # Projection layer
        layers.append(
            ConvBNAct(
                in_channels=hidden_channels,
                out_channels=out_channels,
                kernel_size=1,
                stride=1,
                groups=1,
                activation=nn.Identity,
            )
        )
        
        # Execute layers
        self.block = nn.Sequential(*layers)
    
    def forward(self, x):
        out = self.block(x)
        if self.use_res_connect:
            out = self.drop_path(out)
            out = x + out
        return out


class EfficientNet(nn.Module):
    def __init__(self, in_channels, depth_factor, width_factor, num_classes, dropout):
        super().__init__()
        layers = []
        
        # [expand, kernel_size, stride, out_channels, layers]
        baseline_network = [
            (1, 3, 1, 16, 1),
            (6, 3, 2, 24, 2),
            (6, 5, 2, 40, 2),
            (6, 3, 2, 80, 3),
            (6, 5, 1, 112, 3),
            (6, 5, 2, 192, 4),
            (6, 3, 1, 320, 1),
        ]
        
        # Stage 1
        out_channels = _make_divisible(32 * width_factor)
        layers.append(
            ConvBNAct(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=2
            )
        )
        in_channels = out_channels
        
        # Stage 2 -> 8
        for exp, k, s, out, l in baseline_network:
            out_channels = _make_divisible(out * width_factor)
            new_l = math.ceil(l * depth_factor)
            for i in range(new_l):
                stride = s if i == 0 else 1
                layers.append(
                    MBConv(
                        in_channels=in_channels,
                        out_channels=out_channels,
                        expand_ratio=exp,
                        kernel_size=k,
                        stride=stride,
                    )
                )
                in_channels = out_channels
        
        out_channels = _make_divisible(int(1280 * width_factor), divisor=8)
        layers.append(
            ConvBNAct(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=1,
            )
        )
        
        layers.append(nn.AdaptiveAvgPool2d(1))
        
        self.blocks = nn.Sequential(*layers)
        
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(out_channels, num_classes),
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.blocks(x)
        x = torch.flatten(x, 1)
        logits = self.head(x)
        return logits

# 5. Training Preparation

# 6. Train

# 7. GradCAM